## SelfGNN — Synthetic Merchant Dataset Preprocessing

**Input:** `datasetRaw/synthetic/dataset.csv` (~5M rows)

| File | Description |
|------|-------------|
| `trn_mat_time` | `[global_mat, subMat (T=12), timeMat]` pickle |
| `sequence` | Per-user chronological merchant visit list |
| `tst_int` / `val_int` | Held-out positive merchant per user |
| `test_dict` / `val_dict` | `{uid → [neg_merchants]}` (all available negatives) |
| `user2id.pkl` / `merchant2id.pkl` | ID mappings for feature extraction |
| `train/test/val_synthetic_merchant.csv` | TSV exports |
| `edge_weights.pkl` | `{(uid,mid) → avg_amount}` for `--use_edge_features` |

**Note:** Only ~48 unique merchants → near-complete bipartite graph. `testSize` clamped to `merchantnum-1`.

**Configuration:** `MIN_INTERACTIONS=5` (5-core), `GRAPH_NUM=12` monthly slices, `TEST_NEG=merchantnum-1`

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
OUT_DIR          = '../Datasets/synthetic-merchant/'
RAW_PATH         = '../datasetRaw/synthetic/dataset.csv'
MIN_INTERACTIONS = 5
GRAPH_NUM        = 12   # monthly time slices
PICK_NUM         = 10_000
SEED             = 100

# ── Imports ────────────────────────────────────────────────────────────────────
import os, pickle, copy, ast
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

os.makedirs(OUT_DIR, exist_ok=True)
print('Libraries loaded.')
print('Output dir :', OUT_DIR)
print('Raw path   :', RAW_PATH)

Libraries loaded.
Output dir : ../Datasets/synthetic-merchant/
Raw path   : ../datasetRaw/synthetic/dataset.csv


In [2]:
# ── Cell 2: Load CSV ────────────────────────────────────────────────────────────
chunks = []
for chunk in pd.read_csv(RAW_PATH, chunksize=500_000, low_memory=False):
    chunk = chunk.dropna(subset=['customer_id', 'merchant_name', 'timestamp', 'amount_mnt'])
    chunks.append(chunk[['customer_id', 'merchant_name', 'timestamp', 'amount_mnt']])

df = pd.concat(chunks, ignore_index=True)
df['ts'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['ts'])
df['unix_ts']      = df['ts'].astype(np.int64) // 10**9
df['customer_id']  = df['customer_id'].astype(str)
df['merchant_name'] = df['merchant_name'].astype(str)
df['amount_mnt']   = pd.to_numeric(df['amount_mnt'], errors='coerce').clip(lower=0)
df = df.dropna(subset=['amount_mnt'])

print(f'Loaded     : {len(df):,} rows')
print(f'Users      : {df["customer_id"].nunique():,}')
print(f'Merchants  : {df["merchant_name"].nunique():,}')

Loaded     : 2,001,863 rows
Users      : 99,731
Merchants  : 48


In [3]:
# ── Cell 3: K-core filter ───────────────────────────────────────────────────────
def kcore_filter_df(df: pd.DataFrame, user_col: str, item_col: str, k: int = MIN_INTERACTIONS) -> pd.DataFrame:
    for iteration in range(1000):
        u_deg = df[user_col].value_counts()
        i_deg = df[item_col].value_counts()
        valid_u = set(u_deg[u_deg >= k].index)
        valid_i = set(i_deg[i_deg >= k].index)
        new_df = df[df[user_col].isin(valid_u) & df[item_col].isin(valid_i)]
        print(f'  Iter {iteration+1}: {new_df[user_col].nunique():,} users, '
              f'{new_df[item_col].nunique():,} merchants, {len(new_df):,} edges')
        if len(new_df) == len(df):
            break
        df = new_df.reset_index(drop=True)
    return df

print(f'K-core filtering (k={MIN_INTERACTIONS})...')
df_filt = kcore_filter_df(df, 'customer_id', 'merchant_name')
print(f'After k-core: {df_filt["customer_id"].nunique():,} users, '
      f'{df_filt["merchant_name"].nunique():,} merchants')

K-core filtering (k=5)...
  Iter 1: 81,433 users, 48 merchants, 1,944,655 edges
  Iter 2: 81,433 users, 48 merchants, 1,944,655 edges
After k-core: 81,433 users, 48 merchants


In [4]:
# ── Cell 4: ID mapping + interaction dict + minn/maxx + edge_weights ───────────
user_strs     = sorted(df_filt['customer_id'].unique())
merchant_strs = sorted(df_filt['merchant_name'].unique())

user2id     = {u: i for i, u in enumerate(user_strs)}
merchant2id = {m: i for i, m in enumerate(merchant_strs)}

usrnum      = len(user2id)
merchantnum = len(merchant2id)
TEST_NEG    = merchantnum - 1   # use all available negatives

df_filt = df_filt.copy()
df_filt['uid'] = df_filt['customer_id'].map(user2id)
df_filt['mid'] = df_filt['merchant_name'].map(merchant2id)

with open(OUT_DIR + 'user2id.pkl', 'wb') as f:
    pickle.dump(user2id, f)
with open(OUT_DIR + 'merchant2id.pkl', 'wb') as f:
    pickle.dump(merchant2id, f)
print(f'Saved: user2id.pkl ({usrnum:,}), merchant2id.pkl ({merchantnum:,})')

# Build integer-keyed interaction dict
interaction: list = [defaultdict(list) for _ in range(usrnum)]
for row in df_filt[['uid', 'mid', 'unix_ts']].itertuples(index=False):
    interaction[row.uid][row.mid].append(row.unix_ts)
for uid in range(usrnum):
    for mid in interaction[uid]:
        interaction[uid][mid] = sorted(interaction[uid][mid])

minn = int(df_filt['unix_ts'].min())
maxx = int(df_filt['unix_ts'].max())

n_pairs  = df_filt.groupby(['uid', 'mid']).ngroups
n_events = len(df_filt)
density  = n_pairs / (usrnum * merchantnum) * 100

print(f'usrnum      = {usrnum:,}')
print(f'merchantnum = {merchantnum:,}  ← few merchants (near-complete graph)')
print(f'Unique pairs: {n_pairs:,} | Total events: {n_events:,}')
print(f'Density     : {density:.2f}%')
print(f'Time range  : {pd.Timestamp(minn, unit="s")} → {pd.Timestamp(maxx, unit="s")}')

# Edge weights: average amount per (uid, mid) pair
ew_sum: dict = defaultdict(float)
ew_cnt: dict = defaultdict(int)
for row in df_filt[['uid', 'mid', 'amount_mnt']].itertuples(index=False):
    k = (int(row.uid), int(row.mid))
    ew_sum[k] += float(row.amount_mnt)
    ew_cnt[k] += 1
edge_weights = {k: ew_sum[k] / ew_cnt[k] for k in ew_cnt}
print(f'Edge weights: {len(edge_weights):,} pairs')

Saved: user2id.pkl (81,433), merchant2id.pkl (48)
usrnum      = 81,433
merchantnum = 48  ← few merchants (near-complete graph)
Unique pairs: 946,666 | Total events: 1,944,655
Density     : 24.22%
Time range  : 1970-01-20 08:35:32 → 1970-01-21 02:06:43
Edge weights: 946,666 pairs


In [5]:
# ── Cell 5: Dataset statistics ──────────────────────────────────────────────────
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    user_degrees     = [len(interaction[u]) for u in range(usrnum) if interaction[u]]
    merchant_deg_cnt: dict = defaultdict(int)
    for u in range(usrnum):
        for mid in interaction[u]:
            merchant_deg_cnt[mid] += 1
    merchant_degrees = list(merchant_deg_cnt.values())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Synthetic-Merchant Dataset Statistics', fontweight='bold')

    axes[0].hist(user_degrees, bins=50, color='#1976D2', alpha=0.8)
    axes[0].set_xlabel('Merchants per user')
    axes[0].set_ylabel('Count')
    axes[0].set_title('User degree distribution')
    axes[0].set_yscale('log')

    axes[1].hist(merchant_degrees, bins=min(50, merchantnum), color='#F57C00', alpha=0.8)
    axes[1].set_xlabel('Users per merchant')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Merchant degree distribution')
    axes[1].set_yscale('log')

    plt.tight_layout()
    plt.savefig(OUT_DIR + 'dataset_stats.png', bbox_inches='tight', dpi=120)
    plt.show()
    print('Saved: dataset_stats.png')
except ImportError:
    print('matplotlib not available, skipping plot')

print(f'Users         : {usrnum:>10,}')
print(f'Merchants     : {merchantnum:>10,}')
print(f'Unique pairs  : {n_pairs:>10,}')
print(f'Total events  : {n_events:>10,}')
print(f'Density       : {density:>10.2f}%')

Saved: dataset_stats.png
Users         :     81,433
Merchants     :         48
Unique pairs  :    946,666
Total events  :  1,944,655
Density       :      24.22%


/var/folders/sr/2htv7_wj6sj6qtmjtk9f1vpc0000gn/T/ipykernel_26429/266181873.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ── Cell 6: Train / test / val split ───────────────────────────────────────────

def negSamp(pos_set: set, samp_size: int, node_num: int) -> list:
    valid = [j for j in range(node_num) if j not in pos_set]
    if not valid:
        valid = list(range(node_num))
    samp_size = min(samp_size, len(valid))
    idx = np.random.choice(len(valid), size=samp_size, replace=False)
    return [valid[i] for i in idx]


def split_test_val(
    interaction: list,
    usrnum: int,
    merchantnum: int,
    test_neg: int = TEST_NEG,
    pick_num: int = PICK_NUM,
    seed: int = SEED,
) -> tuple:
    np.random.seed(seed)
    eligible = [
        u for u in range(usrnum)
        if interaction[u] and len(interaction[u]) >= 3
    ]
    perm     = np.random.permutation(eligible)
    pick_usr = perm[:pick_num]

    tst_int   = [None] * usrnum
    val_int   = [None] * usrnum
    test_rows: list = []
    val_rows:  list = []
    skipped = 0

    for u in pick_usr:
        merch = interaction[u]
        if not merch or len(merch) < 3:
            skipped += 1
            continue

        pairs = sorted((ts, mid) for mid, tss in merch.items() for ts in tss)

        tst_time, tst_mid = pairs[-1]
        val_mid = val_time = None
        for ts, mid in reversed(pairs[:-1]):
            if mid != tst_mid:
                val_mid, val_time = mid, ts
                break
        if val_mid is None:
            skipped += 1
            continue

        # Remove only the held-out timestamp
        tss = [t for t in interaction[u][tst_mid] if t != tst_time]
        if tss:
            interaction[u][tst_mid] = tss
        else:
            del interaction[u][tst_mid]

        tss = [t for t in interaction[u].get(val_mid, []) if t != val_time]
        if tss:
            interaction[u][val_mid] = tss
        elif val_mid in interaction[u]:
            del interaction[u][val_mid]

        if sum(1 for tss in interaction[u].values() if tss) < 1:
            skipped += 1
            continue

        tst_int[u] = tst_mid
        val_int[u] = val_mid
        all_pos    = set(merch.keys())

        test_rows.append({'user_id': u+1, 'merchant_id': tst_mid,
                          'time': tst_time, 'neg_merchants': str(negSamp(all_pos, test_neg, merchantnum))})
        val_rows.append( {'user_id': u+1, 'merchant_id': val_mid,
                          'time': val_time,  'neg_merchants': str(negSamp(all_pos, test_neg, merchantnum))})

    pd.DataFrame(test_rows).to_csv(OUT_DIR + 'test_synthetic_merchant.csv', sep='\t', index=False)
    pd.DataFrame(val_rows).to_csv( OUT_DIR + 'val_synthetic_merchant.csv',  sep='\t', index=False)

    n_test = sum(1 for x in tst_int if x is not None)
    n_val  = sum(1 for x in val_int  if x is not None)
    print(f'Test users : {n_test:,}')
    print(f'Val  users : {n_val:,}')
    print(f'Skipped    : {skipped:,}')
    print(f'Negatives per user : {test_neg} (all available merchants except positives)')
    return interaction, tst_int, val_int


interaction_train = copy.deepcopy(interaction)
interaction_train, tstInt, valInt = split_test_val(interaction_train, usrnum, merchantnum)

Test users : 10,000
Val  users : 10,000
Skipped    : 0
Negatives per user : 47 (all available merchants except positives)


In [7]:
# ── Cell 7: Build graph matrices ────────────────────────────────────────────────

def trans(interaction: list, usrnum: int, merchantnum: int) -> csr_matrix:
    r, c, d = [], [], []
    for uid in range(usrnum):
        if not interaction[uid]:
            continue
        for mid, tss in interaction[uid].items():
            for _ in tss:
                r.append(uid); c.append(mid); d.append(1.0)
    mat = csr_matrix((d, (r, c)), shape=(usrnum, merchantnum))
    print(f'Global matrix: {mat.shape}, {(mat!=0).nnz:,} unique pairs, {len(d):,} total events')
    return mat


def trans_sub(
    interaction: list, usrnum: int, merchantnum: int,
    graph_num: int, minn: int, maxx: int,
) -> tuple:
    interval = (maxx - minn) / graph_num
    rcd       = [([],[],[]) for _ in range(graph_num)]
    seen      = [set() for _ in range(graph_num)]
    last_int: dict = {}

    for uid in range(usrnum):
        if not interaction[uid]:
            continue
        for mid, tss in interaction[uid].items():
            for ts in tss:
                t   = max(0, min(int((ts - minn) / interval), graph_num - 1))
                key = (uid, mid)
                if key not in seen[t]:
                    rcd[t][0].append(uid); rcd[t][1].append(mid); rcd[t][2].append(1.0)
                    seen[t].add(key)
                if key not in last_int or last_int[key] < t:
                    last_int[key] = t

    int_mats = []
    for t in range(graph_num):
        mat = csr_matrix((rcd[t][2], (rcd[t][0], rcd[t][1])), shape=(usrnum, merchantnum))
        int_mats.append(mat)
        print(f'  A_{t}: {mat.nnz:,} non-zeros')

    tr, tc, td = [], [], []
    for (u, i), t in last_int.items():
        tr.append(u); tc.append(i); td.append(t)
    time_mat = csr_matrix((td, (tr, tc)), shape=(usrnum, merchantnum))
    print(f'  timeMat: {time_mat.nnz:,} non-zeros')
    return int_mats, time_mat


global_mat = trans(interaction_train, usrnum, merchantnum)
print(f'\nBuilding T={GRAPH_NUM} time-interval sub-graphs...')
subMat, timeMat = trans_sub(interaction_train, usrnum, merchantnum, GRAPH_NUM, minn, maxx)

Global matrix: (81433, 48), 931,969 unique pairs, 1,924,655 total events

Building T=12 time-interval sub-graphs...
  A_0: 123,180 non-zeros
  A_1: 123,103 non-zeros
  A_2: 122,754 non-zeros
  A_3: 122,753 non-zeros
  A_4: 123,182 non-zeros
  A_5: 123,443 non-zeros
  A_6: 122,652 non-zeros
  A_7: 122,126 non-zeros
  A_8: 121,604 non-zeros
  A_9: 120,333 non-zeros
  A_10: 118,991 non-zeros
  A_11: 114,677 non-zeros
  timeMat: 931,969 non-zeros


In [8]:
# ── Cell 8: Build sequence + save all pickles ───────────────────────────────────

sequence: list = []
empty_count = 0
for uid in range(usrnum):
    if not interaction_train[uid]:
        sequence.append([0])
        empty_count += 1
        continue
    pairs = sorted((ts, mid) for mid, tss in interaction_train[uid].items() for ts in tss)
    sequence.append([mid for _, mid in pairs] if pairs else [0])
    if not pairs:
        empty_count += 1

# Consistency assertion
seq_events = sum(len(s) for s in sequence) - empty_count
mat_events = int(global_mat.sum())
assert seq_events == mat_events, (
    f'MISMATCH: sequence has {seq_events:,} events but global_mat has {mat_events:,}'
)
print(f'Sequence events : {seq_events:,}')
print(f'Global mat sum  : {mat_events:,}')
print('Sequence-matrix consistency check PASSED.')

# Save core pickles
with open(OUT_DIR + 'trn_mat_time', 'wb') as f: pickle.dump([global_mat, subMat, timeMat], f)
with open(OUT_DIR + 'tst_int',      'wb') as f: pickle.dump(tstInt, f)
with open(OUT_DIR + 'val_int',      'wb') as f: pickle.dump(valInt, f)
with open(OUT_DIR + 'sequence',     'wb') as f: pickle.dump(sequence, f)
print('Saved: trn_mat_time, tst_int, val_int, sequence')

# test_dict / val_dict
for split, csv_name in [('test', 'test_synthetic_merchant.csv'), ('val', 'val_synthetic_merchant.csv')]:
    df_split = pd.read_csv(OUT_DIR + csv_name, sep='\t')
    d: dict = {}
    for _, row in df_split.iterrows():
        uid  = int(row['user_id']) - 1
        negs = ast.literal_eval(str(row['neg_merchants']))
        d[uid] = negs
    with open(OUT_DIR + f'{split}_dict', 'wb') as f:
        pickle.dump(d, f)
    print(f'Saved: {split}_dict ({len(d):,} users, 0-indexed)')

# Train CSV
train_rows: list = []
for uid in range(usrnum):
    if not interaction_train[uid]:
        continue
    for mid, tss in interaction_train[uid].items():
        for ts in tss:
            train_rows.append({'user_id': uid + 1, 'merchant_id': mid, 'time': ts})
df_train = pd.DataFrame(train_rows)
df_train.to_csv(OUT_DIR + 'train_synthetic_merchant.csv', sep='\t', index=False)
print(f'Saved: train_synthetic_merchant.csv ({len(df_train):,} rows)')

# Edge weights
with open(OUT_DIR + 'edge_weights.pkl', 'wb') as f:
    pickle.dump(edge_weights, f)
print(f'Saved: edge_weights.pkl ({len(edge_weights):,} pairs)')

Sequence events : 1,924,655
Global mat sum  : 1,924,655
Sequence-matrix consistency check PASSED.
Saved: trn_mat_time, tst_int, val_int, sequence
Saved: test_dict (10,000 users, 0-indexed)
Saved: val_dict (10,000 users, 0-indexed)
Saved: train_synthetic_merchant.csv (1,924,655 rows)
Saved: edge_weights.pkl (946,666 pairs)


In [9]:
# ── Cell 9: Verification & summary ─────────────────────────────────────────────

df_test = pd.read_csv(OUT_DIR + 'test_synthetic_merchant.csv', sep='\t')
df_val  = pd.read_csv(OUT_DIR + 'val_synthetic_merchant.csv',  sep='\t')

train_pairs = set(zip(df_train['user_id'], df_train['merchant_id']))
test_pairs  = set(zip(df_test['user_id'],  df_test['merchant_id']))
val_pairs   = set(zip(df_val['user_id'],   df_val['merchant_id']))

overlap_tr_ts = train_pairs & test_pairs
overlap_tr_vl = train_pairs & val_pairs
overlap_ts_vl = test_pairs  & val_pairs

print(f'Train-Test overlap : {len(overlap_tr_ts):,} pairs')
print(f'Train-Val  overlap : {len(overlap_tr_vl):,} pairs')
print(f'Test-Val   overlap : {len(overlap_ts_vl):,} pairs')
if overlap_tr_ts:
    print('  (expected: earlier visits to same merchant remain in train)')

assert len(overlap_ts_vl) == 0, f'FAIL: Test-Val overlap = {len(overlap_ts_vl)} pairs'
print('Test-Val disjointness check PASSED.')

# File inventory
required_files = [
    'trn_mat_time', 'tst_int', 'val_int', 'sequence',
    'test_dict', 'val_dict', 'user2id.pkl', 'merchant2id.pkl',
    'edge_weights.pkl',
    'train_synthetic_merchant.csv', 'test_synthetic_merchant.csv', 'val_synthetic_merchant.csv',
]
print()
print(f'{"File":<35} {"Size"}')
print('-' * 47)
for fname in required_files:
    path   = OUT_DIR + fname
    size   = os.path.getsize(path) if os.path.isfile(path) else None
    status = f'{size / 1024:.0f} KB' if size else 'MISSING'
    print(f'  {fname:<33} {status}')

print()
print('=' * 60)
print('Preprocessing Complete — synthetic-merchant')
print('=' * 60)
print(f'Users                : {usrnum:>10,}')
print(f'Merchants            : {merchantnum:>10,}  ← few (near-complete graph)')
print(f'Train events         : {len(df_train):>10,}')
print(f'Unique train pairs   : {(global_mat != 0).nnz:>10,}')
print(f'Test users           : {len(df_test):>10,}')
print(f'Val  users           : {len(df_val):>10,}')
print(f'Negatives per user   : {TEST_NEG:>10,}')
print(f'Time intervals (T)   : {GRAPH_NUM:>10}')
print(f'Output dir           : {OUT_DIR}')

Train-Test overlap : 2,748 pairs
Train-Val  overlap : 2,555 pairs
Test-Val   overlap : 0 pairs
  (expected: earlier visits to same merchant remain in train)
Test-Val disjointness check PASSED.

File                                Size
-----------------------------------------------
  trn_mat_time                      43394 KB
  tst_int                           89 KB
  val_int                           89 KB
  sequence                          4079 KB
  test_dict                         818 KB
  val_dict                          818 KB
  user2id.pkl                       1463 KB
  merchant2id.pkl                   1 KB
  edge_weights.pkl                  14973 KB
  train_synthetic_merchant.csv      29278 KB
  test_synthetic_merchant.csv       1585 KB
  val_synthetic_merchant.csv        1585 KB

Preprocessing Complete — synthetic-merchant
Users                :     81,433
Merchants            :         48  ← few (near-complete graph)
Train events         :  1,924,655
Unique train pairs 